# 실험 3 재학습 노트북 (Kaggle GPU)

`docs/10-experiment3_investigation.md`의 "다음 단계"를 실행하기 위한 노트북.
Kaggle 세션이 끊겨 `outputs/adapter`가 유실된 상태에서 재학습부터 다시 시작한다.

**진행 순서**: 클론 -> 심플 타깃 데이터 배치 -> 의존성 설치 -> 학습(r=32+MLP) ->
`merge_adapter_plain.py`(순정 peft)로 병합 -> 생성 스모크 테스트 -> 결과 다운로드.

**Kaggle 노트북 설정**:
- Accelerator: GPU T4 x1 (또는 P100)
- Add Input -> Dataset: `simple-data` (이전 세션에서 업로드한
  `data/processed_simple/{train,eval}.jsonl`). 없다면 아래 "데이터 없을 때" 셀 참고.

**커널이 재시작된 경우 (세션 오염 방지, 중요)**:
위에서부터 셀을 다시 실행하지 말 것 (unsloth를 이미 import한 적이 있으면
재import 시 transformers 클래스가 다시 몽키패치되어 검증 결과가 오염될 수 있음).
**"재시작 후에는 여기부터" 마크다운 셀 아래의 `%cd` 셀만 실행**하고, 중단됐던
지점부터 이어서 실행할 것.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 클론 (최초 1회만)

이미 `/kaggle/working/tradecode-lora`가 있으면(재시작 등으로 파일이 남아있는 경우)
다시 클론하지 않고 건너뛴다.

In [ ]:
import os

%cd /kaggle/working
if not os.path.isdir("tradecode-lora"):
    !git clone https://github.com/zynxquzo/tradecode-lora.git
else:
    print("이미 클론되어 있음, 건너뜀")

## 재시작 후에는 여기부터

커널을 재시작했다면 위 셀들(1번까지)은 다시 실행하지 말고, 이 셀만 실행한 뒤
중단됐던 단계로 바로 이동할 것.

In [ ]:
%cd /kaggle/working/tradecode-lora

## 2. 의존성 설치

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. 데이터 배치 (심플 타깃: `{"hs_code": "6402"}` 형태, confidence_basis 없음)

`simple-data` 데이터셋을 Kaggle Input으로 추가했다면 아래에서 그대로 복사한다.
**복사 후 반드시 `head -1`로 `confidence_basis`가 없는 단순화 버전인지 확인할 것**
(예전에 실수로 안 바뀐 채로 학습한 적이 있었음).

In [ ]:
!mkdir -p data/processed
!cp /kaggle/input/simple-data/train.jsonl data/processed/train.jsonl
!cp /kaggle/input/simple-data/eval.jsonl data/processed/eval.jsonl
print("train.jsonl 첫 줄:")
!head -1 data/processed/train.jsonl
print("\neval.jsonl 첫 줄:")
!head -1 data/processed/eval.jsonl

assert "confidence_basis" not in open("data/processed/train.jsonl", encoding="utf-8").readline(), (
    "train.jsonl에 confidence_basis가 남아있음 -- 단순화 버전이 아님, 잘못된 데이터를 복사한 것"
)
print("\n확인 완료: confidence_basis 없음")

### 데이터셋이 Kaggle Input에 없을 때
로컬에서 `data/processed_simple/{train,eval}.jsonl`을 Kaggle Dataset(`simple-data`)으로
새로 업로드하거나, 로컬 `preprocess.py --simple-target`으로 재생성해서 업로드할 것.
(`docs/10-experiment3_investigation.md` 4절 참고)

## 4. 스모크 테스트 (60 step, loss 하강만 확인)

In [ ]:
!python src/finetune/train.py --smoke-test --max-steps 60 \
    --lora-r 32 --lora-alpha 64 \
    --target-modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj

## 5. 본 학습 (약 40분 소요, r=32 + MLP 포함)

In [ ]:
!python src/finetune/train.py \
    --lora-r 32 --lora-alpha 64 \
    --target-modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj

## 6. 병합 (순정 peft -- `merge_adapter.py`(unsloth)가 아니라 이 스크립트를 쓸 것)

unsloth의 `save_pretrained_merged()`가 가중치를 손상시키는 버그가 확인됐으므로
(`docs/10-experiment3_investigation.md` 5절), 학습에 쓴 것과 동일한 4bit 베이스에
순정 `peft.PeftModel.merge_and_unload()`로 병합한다.

In [ ]:
!python src/finetune/merge_adapter_plain.py \
    --adapter-dir outputs/adapter --output-dir outputs/merged_plain

## 7. 생성 스모크 테스트 (병합 모델이 실제로 의미있는 출력을 내는지 확인)

**중요**: 이 셀은 `unsloth`를 import하지 않는다 (unsloth는 import 시점에 transformers
모델 클래스를 전역 몽키패치하므로, 검증은 항상 오염되지 않은 순정 transformers로
해야 실제 병합 모델 품질을 신뢰할 수 있다). 만약 이 노트북 세션에서 이미 학습
단계(5번)를 거쳐 unsloth를 import한 적이 있다면, **커널을 재시작한 뒤** 위
"재시작 후에는 여기부터" 셀부터 다시 실행하고 이 셀로 바로 건너올 것.

In [ ]:
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# train.py의 PROMPT_TEMPLATE과 반드시 동일해야 함 (학습 시 프롬프트와 어긋나면
# 정상 모델도 이상하게 생성될 수 있음)
PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

MERGED_DIR = "outputs/merged_plain"

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=torch.float16, device_map="auto")
model.eval()

samples = [json.loads(line) for line in open("data/processed/train.jsonl", encoding="utf-8")][:5]
samples += [json.loads(line) for line in open("data/processed/eval.jsonl", encoding="utf-8")][:5]

for sample in samples:
    prompt = PROMPT_TEMPLATE.format(instruction=sample["instruction"], input=sample["input"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"기대값: {sample['output']!r}")
    print(f"생성값: {generated!r}")
    print("-" * 60)

**판정 기준**: `{"hs_code": "NNNN"}` 형태의 숫자 코드가 생성되면 성공 -- 정확히
일치하지 않아도 괜찮다(정량 평가는 8단계에서). 반대로 `purpoſe`/`vectorielle`
같은 희귀 단어나 빈 문자열/구두점 반복이 나오면 병합이 또 실패한 것이니, 아래
"병합도 실패했을 때" 셀로 이동할 것.

## 8. 결과 다운로드 (Kaggle Output)

7번 셀에서 정상 출력을 확인했다면, 병합 모델을 로컬로 가져가 GGUF 변환 -> Ollama
등록 -> `eval.jsonl` 전체(405건) 정량 재평가를 로컬에서 진행한다
(`docs/10-experiment3_investigation.md` "다음 단계" 4번 참고).

In [ ]:
!zip -r -0 /kaggle/working/merged_plain.zip outputs/merged_plain outputs/adapter docs/02-training_log.md

## 병합도 실패했을 때 (unsloth 버전 고정 후 재시도)

7번에서 여전히 무의미한 출력이 나오면, unsloth 자체 문제가 아니라 다른 원인일 수
있으니 `requirements-colab.txt`에 unsloth 버전을 과거 정상 동작이 보고된 버전으로
고정한 뒤 `merge_adapter.py`(unsloth 경로)로 재시도한다. 아직 어느 버전부터
회귀가 생겼는지 특정되지 않았으므로, 먼저 GitHub에서 unsloth 릴리스 히스토리를
확인해 2026.8.1 이전 버전을 시험해볼 것.